In [2]:
import mitsuba as mi
mi.set_variant("cuda_ad_mono_polarized")

import numpy as np
import sionna.rt
from sionna.rt import (
    load_scene, PlanarArray,
    Transmitter, Receiver, PathSolver,
)

In [3]:
from pathlib import Path
from copy import deepcopy
import xml.etree.ElementTree as ET

src = Path("public/test-Manhattan.xml")
dst = src.with_name("test-Manhattan-materials.xml")

original = ET.parse(src).getroot()
root = ET.Element("scene", version=original.get("version", "3.0.0"))

# Sionna nhận các ID ITU: nhà dùng gạch, đường/path dùng bê tông nhựa (asphalt)
for material_id in ("mat-itu_brick", "mat-itu_asphalt_concrete"):
    ET.SubElement(root, "bsdf", type="diffuse", id=material_id)

counts = {"buildings": 0, "roads": 0}
skipped = []

for shape in original.findall("shape"):
    filename = shape.find("string[@name='filename']")
    mesh_name = filename.get("value", "") if filename is not None else ""
    label = f"{shape.get('id', '')} {mesh_name}".lower()

    if shape.get("type") not in {"ply", "obj", "serialized"}:
        raise ValueError(
            f"Còn hình học không hỗ trợ: {shape.get('type')} — {label}"
        )

    if "building" in label:
        material_id = "mat-itu_brick"
        counts["buildings"] += 1
    elif "road" in label or "path" in label:
        material_id = "mat-itu_asphalt_concrete"
        counts["roads"] += 1
    else:
        skipped.append(label)
        continue

    new_shape = deepcopy(shape)

    # Với mesh xuất từ Blender: thay vật liệu, giữ hình học và transform
    for child in list(new_shape):
        if child.tag in {"bsdf", "ref", "emitter"}:
            new_shape.remove(child)

    ET.SubElement(new_shape, "ref", name="bsdf", id=material_id)
    root.append(new_shape)

if not all(counts.values()):
    raise ValueError(f"Chưa nhận diện đủ nhà và đường: {counts}")

ET.indent(root)
ET.ElementTree(root).write(
    dst, encoding="utf-8", xml_declaration=True
)

print("Số mesh theo nhóm:", counts)
print("Đối tượng bỏ qua:", skipped)
print("File mới:", dst)

Số mesh theo nhóm: {'buildings': 15, 'roads': 8}
Đối tượng bỏ qua: ['elm__34 meshes/map_osm_areas_pedestrian.ply', 'elm__52 meshes/map_osm_vegetation.ply', 'elm__54 meshes/map_osm_water.ply']
File mới: public/test-Manhattan-materials.xml


In [4]:
scene = load_scene("public/test-Manhattan-materials.xml")

In [5]:
for name, obj in scene.objects.items():
    print(name, "→", obj.radio_material.itu_type)

no-name-1 → asphalt_concrete
no-name-2 → brick


In [6]:
scene.preview(point_picker=True)

### Init frequency

In [7]:
scene.frequency = 3.5e9

### Init one anten for each station

In [8]:
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V",
)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V",
)

### Init latitude & longtitude for each station

In [9]:
scene.add(Transmitter(name="tx", position=[-110.653, -73.533, 5.800 + 1.5]))
scene.add(Receiver(name="rx", position=[-23.614, 136.679, 5.800 + 1.5]))

In [10]:
solver = PathSolver()
paths = solver(
    scene=scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False,
    seed=42,
)

In [11]:
a, tau = paths.cir(
    normalize_delays=False,
    out_type="numpy",
)

# Chọn liên kết Rx0–Tx0, anten 0, thời điểm 0
coefficients = a[0, 0, 0, 0, :, 0]
delays = tau[0, 0, :] 

valid = (delays >= 0) & (np.abs(coefficients) > 0)
print("Số đường truyền tìm được:", np.count_nonzero(valid))

for index in np.flatnonzero(valid):
    print(
        f"Path {index}: "
        f"delay={delays[index] * 1e9:.2f} ns, "
        f"|a|={abs(coefficients[index]):.3e}"
    )

Số đường truyền tìm được: 10
Path 0: delay=758.92 ns, |a|=2.996e-05
Path 1: delay=984.58 ns, |a|=6.877e-06
Path 2: delay=826.56 ns, |a|=1.294e-05
Path 3: delay=808.40 ns, |a|=1.319e-05
Path 4: delay=962.03 ns, |a|=6.884e-06
Path 5: delay=1167.75 ns, |a|=3.448e-06
Path 6: delay=984.58 ns, |a|=6.877e-06
Path 7: delay=984.58 ns, |a|=6.877e-06
Path 8: delay=1213.84 ns, |a|=3.302e-06
Path 9: delay=827.88 ns, |a|=1.076e-05


In [12]:
scene.preview(paths=paths)